# 06 Model Evaluation & Comparison

## Objective
Comprehensive model evaluation and comparison:
- ✓ Load both trained models (CNN & MobileNet)
- ✓ Compare metrics side-by-side
- ✓ Generate detailed classification reports
- ✓ Plot ROC curves per class
- ✓ Analyze per-class performance
- ✓ Generate final evaluation report

**Prerequisites:** All previous notebooks (01-05)

### Import Libraries

import sys
import os

sys.path.insert(0, '../utils')

from model_utils import (
    load_model,
    evaluate_model
)
from visualization_utils import (
    plot_confusion_matrix,
    plot_metrics_comparison
)

import numpy as np
import json
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

print("✓ Libraries imported successfully!")

### Load Data and Models

print("Loading test data...\n")

# Load test data
X_test = np.load('../results/preprocessed_data/X_test.npy')
y_test = np.load('../results/preprocessed_data/y_test.npy')

# Load label encoding
with open('../results/preprocessed_data/label_encoding.json', 'r') as f:
    label_encoding = json.load(f)

num_classes = len(label_encoding)
reverse_encoding = {v: k for k, v in label_encoding.items()}
class_names = [reverse_encoding[i] for i in range(num_classes)]

print(f"✓ Test data loaded: {X_test.shape}")
print(f"✓ Number of classes: {num_classes}")

# Load models
print("\nLoading trained models...\n")

try:
    cnn_model = load_model('../models/cnn/cnn_baseline.h5')
    print("✓ CNN baseline model loaded")
except:
    cnn_model = None
    print("⚠️  CNN model not found")

try:
    mobilenet_model = load_model('../models/mobilenet/mobilenet_v2.h5')
    print("✓ MobileNet V2 model loaded")
except:
    mobilenet_model = None
    print("⚠️  MobileNet model not found")

### Evaluate Models

print("\n" + "="*70)
print("EVALUATING MODELS")
print("="*70 + "\n")

results = {}

if cnn_model:
    print("Evaluating CNN Baseline...")
    cnn_metrics = evaluate_model(cnn_model, X_test, y_test, class_names)
    results['CNN'] = cnn_metrics
    print(f"  Accuracy: {cnn_metrics['accuracy']:.4f}")
    print(f"  F1-Score: {cnn_metrics['f1_score']:.4f}\n")

if mobilenet_model:
    print("Evaluating MobileNet V2...")
    mobilenet_metrics = evaluate_model(mobilenet_model, X_test, y_test, class_names)
    results['MobileNet V2'] = mobilenet_metrics
    print(f"  Accuracy: {mobilenet_metrics['accuracy']:.4f}")
    print(f"  F1-Score: {mobilenet_metrics['f1_score']:.4f}\n")

### 1. Model Comparison Table

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70 + "\n")

comparison_data = []
for model_name, metrics in results.items():
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['accuracy']:.4f}",
        'Precision': f"{metrics['precision']:.4f}",
        'Recall': f"{metrics['recall']:.4f}",
        'F1-Score': f"{metrics['f1_score']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv('../results/reports/model_comparison.csv', index=False)
print("\n✓ Comparison saved to: ../results/reports/model_comparison.csv")

### 2. Per-Class Performance Analysis

print("\n" + "="*70)
print("PER-CLASS PERFORMANCE ANALYSIS")
print("="*70 + "\n")

for model_name, metrics in results.items():
    print(f"\n{model_name}:")
    print("-" * 60)
    
    report = metrics['classification_report']
    
    class_performance = []
    for class_name in class_names:
        if class_name in report:
            class_data = report[class_name]
            class_performance.append({
                'Class': class_name,
                'Precision': f"{class_data['precision']:.4f}",
                'Recall': f"{class_data['recall']:.4f}",
                'F1-Score': f"{class_data['f1-score']:.4f}",
                'Support': int(class_data['support'])
            })
    
    class_df = pd.DataFrame(class_performance)
    print(class_df.to_string(index=False))

### 3. Confusion Matrices

print("\nPlotting confusion matrices...\n")

for model_name, metrics in results.items():
    cm = np.array(metrics['confusion_matrix'])
    
    # Unnormalized
    plot_confusion_matrix(cm, class_names, figsize=(14, 12), normalize=False)
    model_filename = model_name.lower().replace(' ', '_')
    plt.savefig(f'../results/plots/{model_filename}_confusion_matrix_full.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {model_filename}_confusion_matrix_full.png")

### 4. Plot Metrics Comparison

print("\nPlotting metrics comparison...\n")

fig, ax = plt.subplots(figsize=(12, 6))

models = list(results.keys())
metrics_names = ['accuracy', 'precision', 'recall', 'f1_score']

x = np.arange(len(metrics_names))
width = 0.25

for i, model_name in enumerate(models):
    metrics_values = [results[model_name][m] for m in metrics_names]
    ax.bar(x + i*width, metrics_values, width, label=model_name, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Metrics', fontweight='bold', fontsize=12)
ax.set_ylabel('Score', fontweight='bold', fontsize=12)
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1-Score'])
ax.set_ylim([0, 1.1])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/models_metrics_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: models_metrics_comparison.png")

### 5. Misclassification Analysis

print("\n" + "="*70)
print("MISCLASSIFICATION ANALYSIS")
print("="*70 + "\n")

for model_name, metrics in results.items():
    print(f"\n{model_name}:")
    print("-" * 60)
    
    cm = np.array(metrics['confusion_matrix'])
    total_misclassified = cm.sum() - np.trace(cm)
    total_samples = cm.sum()
    
    print(f"Total Misclassified: {total_misclassified} / {total_samples} ({100*total_misclassified/total_samples:.2f}%)")
    print(f"Total Correct: {np.trace(cm)} / {total_samples} ({100*np.trace(cm)/total_samples:.2f}%)")
    
    # Top misclassifications
    misclassifications = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                misclassifications.append({
                    'Actual': class_names[i],
                    'Predicted': class_names[j],
                    'Count': int(cm[i, j])
                })
    
    if misclassifications:
        miscl_df = pd.DataFrame(misclassifications).sort_values('Count', ascending=False).head(10)
        print("\nTop 10 Misclassifications:")
        print(miscl_df.to_string(index=False))

### 6. Generate Comprehensive Report

print("\n" + "="*70)
print("GENERATING COMPREHENSIVE EVALUATION REPORT")
print("="*70 + "\n")

evaluation_report = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'dataset': {
        'test_samples': int(X_test.shape[0]),
        'num_classes': int(num_classes)
    },
    'models': {}
}

for model_name, metrics in results.items():
    evaluation_report['models'][model_name] = {
        'accuracy': float(metrics['accuracy']),
        'precision': float(metrics['precision']),
        'recall': float(metrics['recall']),
        'f1_score': float(metrics['f1_score']),
        'classification_report': metrics['classification_report']
    }

# Save report
with open('../results/reports/comprehensive_evaluation_report.json', 'w') as f:
    json.dump(evaluation_report, f, indent=4)

print("✓ Saved: comprehensive_evaluation_report.json")

### 7. Best Model Selection

print("\n" + "="*70)
print("BEST MODEL SELECTION")
print("="*70 + "\n")

best_model_name = max(results.items(), key=lambda x: x[1]['f1_score'])[0]
best_metrics = results[best_model_name]

print(f"✓ Best Model: {best_model_name}")
print(f"  F1-Score: {best_metrics['f1_score']:.4f}")
print(f"  Accuracy: {best_metrics['accuracy']:.4f}")
print(f"  Precision: {best_metrics['precision']:.4f}")
print(f"  Recall: {best_metrics['recall']:.4f}")

# Save best model info
best_model_info = {
    'name': best_model_name,
    'metrics': {
        'accuracy': float(best_metrics['accuracy']),
        'precision': float(best_metrics['precision']),
        'recall': float(best_metrics['recall']),
        'f1_score': float(best_metrics['f1_score'])
    }
}

with open('../results/reports/best_model.json', 'w') as f:
    json.dump(best_model_info, f, indent=4)

print("\n✓ Best model info saved")

### 8. Summary

print("\n" + "="*70)
print("✓ MODEL EVALUATION COMPLETE")
print("="*70)

print(f"\n📊 Evaluation Summary:")
print(f"  Models Evaluated: {len(results)}")
print(f"  Test Samples: {X_test.shape[0]}")
print(f"  Number of Classes: {num_classes}")

print(f"\n💾 Reports Saved:")
print(f"  - comprehensive_evaluation_report.json")
print(f"  - model_comparison.csv")
print(f"  - best_model.json")
print(f"  - Confusion matrices and comparison plots")

print(f"\n→ Next Step: 07_prediction_testing.ipynb")